[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/multi_pav_loft.ipynb)

# Exercício resolvido: Pavimentos com Loft
## Fernando Ferraz Rbeiro




### Instalação dos pacotes

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install build123d ezdxf
    !wget -q -N https://raw.githubusercontent.com/255ribeiro/cadquery_basics/master/docs/tuto_colab_build/build123d_simpleviewer.py
else:
    print("Not running in Colab, skipping package installation.")

### Importação dos pacotes

In [ ]:
from build123d import *
from cadquery_simple_viewer import show
import numpy as np
import ezdxf

### criação de volumes no build123d

In [ ]:
# ── Parâmetros ───────────────────────────────────────────────────────────────
cota_inicial  = 0
pap           = 18
n_pav         = 5
rot_inc       = 5
smooth_factor = 1 / (np.pi * 2)


# ══════════════════════════════════════════════════════════════════════════════
# VERSÃO 1 — Rectangle com scale e rotação
# ══════════════════════════════════════════════════════════════════════════════
print('loop start')
sections = []
for i in range(n_pav + 1):
    cota_atual   = round(cota_inicial + pap * i, 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)
    print(f'section {i} start')

    section = (
        Rectangle(30 * scale_factor, 40 * scale_factor)
        .rotate(Axis.Z, i * rot_inc)          # rotação da seção
        .translate((0, 0, cota_atual))        # posiciona na cota
    )
    print(f'section {i} added')

    sections.append(section)

result_rect = loft(sections)


In [ ]:
show(result_rect, tessellation_tolerance=.2)

In [ ]:
# ── Parâmetros ───────────────────────────────────────────────────────────────
cota_inicial  = 0
pap           = 18
n_pav         = 5
rot_inc       = 10
smooth_factor = 1 / (np.pi * 2)

# ══════════════════════════════════════════════════════════════════════════════
# VERSÃO 2 — perfil DXF com scale e rotação
# ══════════════════════════════════════════════════════════════════════════════
def import_dxf_layer(path, layer):
    """
    Importa apenas as entidades de uma layer específica de um DXF.

    O import_dxf() do build123d lê todas as entidades do modelspace de uma
    vez, sem opção de filtrar por layer (diferente do
    cq.importers.importDXF(..., include=[...]) do CadQuery). Por isso
    usamos o ezdxf para copiar só a layer desejada para um DXF temporário
    antes de importar.
    """
    doc = ezdxf.readfile(path)
    msp = doc.modelspace()

    filtered_doc = ezdxf.new(dxfversion=doc.dxfversion)
    filtered_msp = filtered_doc.modelspace()
    for entity in msp.query(f'*[layer=="{layer}"]'):
        filtered_msp.add_foreign_entity(entity)

    tmp_path = f"_layer_{layer}.dxf"
    filtered_doc.saveas(tmp_path)
    return import_dxf(tmp_path)


perfil      = import_dxf_layer("perfil_autocad.dxf", "profile_01")
face_perfil = make_face(perfil)
# perfil_step = scale(import_step("perfil.step"), by=0.001)   # STEP from Rhino
# face_perfil = make_face(perfil_step.wires())

print('loop start')
sections = []
for i in range(n_pav + 1):
    cota_atual   = round(cota_inicial + pap * i, 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)
    print(f'start section {i}')

    section = (
        scale(face_perfil, by=(scale_factor, scale_factor, 1))
        .rotate(Axis.Z, i * rot_inc)          # rotação da seção
        .translate((0, 0, cota_atual))        # posiciona na cota
    )
    print(f'section {i} added')

    sections.append(section)

result_dxf = loft(sections)

In [ ]:
show(result_dxf)